In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- expose_isin ---
FIX_EXPOSE_ISIN_DATA = pd.DataFrame({"id":[1,2,3],"value":[10,20,30]})

self = SimpleNamespace(data=pd.DataFrame({"pol_num": [1, 2, 3], "val": [10, 20, 30]}))
print("✅ Fixtures loaded")
by = ["pol_num"]


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_expose_isin(data):
    assert all(pd.Series(by).isin(self.data.columns))
    return None

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_expose_isin(data):

    assert all(pl.Series(by).is_in(self.data.columns))
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: expose_isin ===

# L1 smoke – generated
try:
    _r = gen_expose_isin(FIX_EXPOSE_ISIN_DATA)
    print("✅ L1 smoke gen_expose_isin: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_expose_isin: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_expose_isin(FIX_EXPOSE_ISIN_DATA)
    print("✅ L1 smoke before_expose_isin: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_expose_isin: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – both sides should accept the same `by` columns.
try:
    before_expose_isin(FIX_EXPOSE_ISIN_DATA)
    gen_expose_isin(FIX_EXPOSE_ISIN_DATA)
    print("✅ L2 equivalence expose_isin: MATCH")
except Exception as _e:
    print(f"❌ L2 equivalence expose_isin: setup error — {type(_e).__name__}: {_e}")

# L3 edge – missing required by-column should be rejected by both sides.
try:
    _old_self = self
    self = SimpleNamespace(data=pd.DataFrame({"other": [1, 2]}))
    _before_exc = _gen_exc = None
    try:
        before_expose_isin(FIX_EXPOSE_ISIN_DATA)
    except Exception as _e:
        _before_exc = type(_e).__name__
    try:
        gen_expose_isin(FIX_EXPOSE_ISIN_DATA)
    except Exception as _e:
        _gen_exc = type(_e).__name__
    self = _old_self
    if _before_exc == _gen_exc == "AssertionError":
        print("✅ L3 edge expose_isin missing by column: MATCH (AssertionError)")
    else:
        print(f"❌ L3 edge expose_isin missing by column: MISMATCH — before={_before_exc}, gen={_gen_exc}")
except Exception as _e:
    try:
        self = _old_self
    except Exception:
        pass
    print(f"❌ L3 edge expose_isin: {type(_e).__name__}: {_e}")
